# Partie III : Modélisation de Séquences — Fondements Probabilistes du Langage



## 1. L'Objectif Probabiliste d'un Modèle de Langage

### Définition Fondamentale
En traitement automatique du langage naturel (NLP), un **modèle de langage** est un système statistique ou génératif qui cherche à appréhender la structure intrinsèque d'une langue en lui attribuant une mesure de probabilité. L'objectif fondamental peut se diviser en deux tâches complémentaires :

1. **Évaluer la plausibilité d'une séquence :** Calculer la probabilité globale $P(W)$ qu'une séquence de tokens donnée $W = (w_1, w_2, \dots, w_T)$ appartienne à la langue cible (par exemple, déterminer qu'une phrase syntaxiquement et sémantiquement correcte a une probabilité plus élevée qu'une suite de mots aléatoires).
2. **Prédire la suite du contexte :** Estimer la probabilité conditionnelle qu'un token spécifique $w_t$ apparaisse immédiatement après une suite de tokens précédents, notée $w_{<t} = (w_1, w_2, \dots, w_{t-1})$.

### Formulations Mathématiques
Si l'on définit un dictionnaire ou vocabulaire fini $\mathcal{V}$, une séquence ou "phrase" $W$ est un élément de l'ensemble de toutes les séquences possibles $\mathcal{V}^*$. Un modèle de langage parfait définirait une distribution de probabilité sur $\mathcal{V}^*$ telle que :

$$\sum_{W \in \mathcal{V}^*} P(W) = 1 \quad \text{et} \quad P(W) \ge 0$$

Dans le cadre de l'apprentissage supervisé (ou auto-supervisé), l'objectif d'entraînement consiste à maximiser la **log-vraisemblance** (Log-Likelihood) sur un corpus d'entraînement composé de $N$ séquences :

$$\mathcal{L}(\theta) = \sum_{i=1}^{N} \log P(W^{(i)} ; \theta)$$

Où $\theta$ représente l'ensemble des paramètres du modèle (poids des matrices d'un réseau récurrent par exemple).

---



## La Factorisation d'une Séquence par la Règle de Chaîne

### Le Problème de la Dimensionnalité
Calculer directement la probabilité conjointe $P(w_1, w_2, \dots, w_T)$ de manière naïve est impossible sur un texte réel. Si le vocabulaire contient $|\mathcal{V}|$ mots et que la séquence est de longueur $T$, il y aurait $|\mathcal{V}|^T$ combinaisons possibles. Il est statistiquement impossible d'observer toutes ces combinaisons dans un corpus pour en estimer la fréquence.

### Application de la Règle de Chaîne (Chain Rule)
Pour contourner cette limite, on utilise un théorème fondamental des probabilités : la **règle de chaîne**. Elle permet de décomposer n'importe quelle probabilité conjointe en un produit de probabilités conditionnelles successives. 

Pour une séquence de longueur $T$, la factorisation exacte s'écrit :

$$P(w_1, w_2, w_3, \dots, w_T) = P(w_1) \times P(w_2 \mid w_1) \times P(w_3 \mid w_1, w_2) \times \dots \times P(w_T \mid w_1, w_2, \dots, w_{T-1})$$

En utilisant le symbole de produit ($\prod$), la formule compacte devient :

$$P(W) = \prod_{t=1}^{T} P(w_t \mid w_1, w_2, \dots, w_{t-1}) = \prod_{t=1}^{T} P(w_t \mid w_{<t})$$

Chaque terme $P(w_t \mid w_{<t})$ représente la prédiction du prochain token sachant tout l'historique de gauche.

### Exemple Concret d'Illustration
Prenons la phrase suivante : **"Le chat boit du lait"**
* $w_1 =$ "Le"
* $w_2 =$ "chat"
* $w_3 =$ "boit"
* $w_4 =$ "du"
* $w_5 =$ "lait"

Par la règle de chaîne, la probabilité d'occurrence de cette phrase exacte est rigoureusement égale à :

$$P(\text{"Le chat boit du lait"}) = P(\text{"Le"}) \times P(\text{"chat"} \mid \text{"Le"}) \times P(\text{"boit"} \mid \text{"Le chat"}) \times P(\text{"du"} \mid \text{"Le chat boit"}) \times P(\text{"lait"} \mid \text{"Le chat boit du"})$$

Le modèle apprend donc à chaque étape temporelle $t$ à distribuer des scores de probabilité sur tout le vocabulaire $\mathcal{V}$ en se basant sur le contexte historique.

---



## Implications pour les Architectures Récurrentes (RNN)

La règle de chaîne impose théoriquement de conditionner la prédiction sur *tout* l'historique $w_{<t}$. L'évolution des architectures de modèles de langage montre comment cette dépendance historique a été traitée :

### L'Approche Historique (Modèles n-grammes)
Faute de capacités de calcul, les modèles traditionnels appliquaient l'**hypothèse de Markov d'ordre $n-1$**, postulant que le mot actuel ne dépend que des $n-1$ mots précédents :

$$P(w_t \mid w_1, \dots, w_{t-1}) \approx P(w_t \mid w_{t-n+1}, \dots, w_{t-1})$$

*Limite :* Perte immédiate du contexte lointain (si $n=3$, le modèle oublie le début d'une phrase dès le 4ème mot).

### L'Approche Récurrente (RNN, LSTM, GRU)
Les réseaux de neurones récurrents éliminent l'approximation markovienne stricte. Ils maintiennent un **état caché** ($h_t$) qui agit comme une mémoire dynamique de tout l'historique.

1. **Compression du contexte :** Au lieu de stocker explicitement la liste des mots $(w_1, \dots, w_{t-1})$, le réseau applique une fonction de transition de manière récursive :
   $$h_t = f(h_{t-1}, w_t)$$
2. **Estimation de la probabilité :** Le vecteur $h_{t-1}$ est censé encapsuler toute l'information sémantique et syntaxique accumulée depuis le début de la séquence. La probabilité conditionnelle est alors modélisée par une couche Dense suivie d'une fonction d'activation `Softmax` appliquée à cet état caché :
   $$P(w_t \mid w_1, \dots, w_{t-1}) \approx P(w_t \mid h_{t-1})$$

C'est cette élégante adéquation mathématique entre la factorisation par la règle de chaîne et la nature séquentielle pas-à-pas des réseaux récurrents qui a fait le succès de ces architectures pour la modélisation probabiliste du langage.
"""

with open("explication_objectif_probabiliste_lm.md", "w", encoding="utf-8") as f:
    f.write(content)
print("File successfully created.")

## 2. La Notion de Perplexité et son Interprétation

### Définition Conceptuelle de la Perplexité
En modélisation du langage, la **perplexité** est la métrique standard utilisée pour évaluer la qualité d'un modèle probabiliste. Conceptuellement, elle mesure le degré de "surprise" ou d'incertitude d'un modèle lorsqu'il lit ou tente de prédire un nouvel échantillon de texte. 

Plus un modèle comprend la syntaxe, la sémantique et le vocabulaire d'une langue, moins il sera surpris par l'apparition des mots composant une phrase réelle, et plus sa perplexité sera basse.

---

### Formulation Mathématique
Mathématiquement, la perplexité (souvent notée $PP$) d'une séquence de tokens $W = (w_1, w_2, \dots, w_N)$ est définie comme l'inverse de la probabilité de cette séquence, normalisée par le nombre de tokens $N$.

En utilisant la règle de la chaîne, la perplexité s'écrit :

$$PP(W) = P(w_1, w_2, \dots, w_N)^{-\frac{1}{N}} = \sqrt[N]{\frac{1}{P(w_1, w_2, \dots, w_N)}}$$

En décomposant par les probabilités conditionnelles :

$$PP(W) = \sqrt[N]{\prod_{i=1}^{N} \frac{1}{P(w_i \mid w_1, \dots, w_{i-1})}}$$

**Le lien avec l'Entropie Croisée (Cross-Entropy) :**
Dans la pratique, multiplier de minuscules probabilités entre elles cause un sous-dépassement numérique (underflow) sur les ordinateurs. On calcule donc toujours la perplexité à partir de la fonction de coût d'**entropie croisée** (Cross-Entropy Loss), notée $H$. La perplexité est simplement l'exponentielle de cette perte :

$$PP(W) = e^{H(W)}$$

---

### Interprétation Concrète d'une Valeur de Perplexité
L'interprétation de la perplexité se comprend de manière intuitive à travers la notion de **facteur de branchement** (branching factor). 

* **Le principe :** Une perplexité de $K$ signifie qu'à chaque étape temporelle (pour prédire le mot suivant), le modèle est en moyenne aussi incertain que s'il devait choisir au hasard et de manière uniforme parmi $K$ mots possibles.
* **Exemple :** Si un modèle a une perplexité de $20$ sur un texte, cela signifie qu'à chaque mot, il "hésite" entre $20$ mots candidats viables.

#### Cas limites pour bien comprendre :
1. **Le modèle parfait (Certitude absolue) :** Si le modèle prédit toujours le mot exact suivant avec une probabilité de $100\%$, sa perplexité sera de $1$. Il n'y a aucune incertitude.
2. **Le modèle aléatoire (Pire cas) :** Si le modèle ne sait rien et choisit le prochain mot totalement au hasard dans un vocabulaire de taille $|\mathcal{V}|$, sa perplexité sera exactement égale à $|\mathcal{V}|$.

---

### Pourquoi utiliser la Perplexité plutôt que l'Exactitude (Accuracy) ?
Contrairement à des tâches de classification classiques (ex: différencier une image de chien d'un chat), le langage naturel est fondamentalement ambigu et ouvert. Après l'amorce *"Je vais manger une"*, les mots *"pomme"*, *"crêpe"*, ou *"pizza"* sont tous corrects. 

* **L'Exactitude (Accuracy)** pénaliserait le modèle à $100\%$ s'il prédit *"crêpe"* alors que la phrase cible disait *"pomme"*. C'est une métrique trop stricte et binaire.
* **La Perplexité** évalue la distribution entière des probabilités. Elle récompense le modèle s'il a attribué une probabilité raisonnablement élevée au mot *"pomme"*, même si sa prédiction "Top 1" était *"crêpe"*. Elle reflète ainsi beaucoup mieux la fluidité et la pertinence du langage généré.

In [ ]:
import torch
import torch.nn as nn


In [ ]:

# Définition de l'appareil d'exécution (basculera naturellement sur le CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:

# ==========================================
# 1. Implémentation d'un RNN Simple
# ==========================================
class ModeleRNNSimple(nn.Module):
    def __init__(self, taille_vocabulaire, dim_embedding, dim_cachee, dim_sortie):
        super(ModeleRNNSimple, self).__init__()
        self.dim_cachee = dim_cachee
        
        # Couche d'encodage (Transformation des tokens en vecteurs denses)
        self.embedding = nn.Embedding(taille_vocabulaire, dim_embedding)
        
        # Couche RNN (batch_first=True signifie que l'entrée est de forme [batch, seq, feature])
        self.rnn = nn.RNN(dim_embedding, dim_cachee, batch_first=True)
        
        # Couche linéaire de classification finale
        self.fc = nn.Linear(dim_cachee, dim_sortie)

    def forward(self, x):
        # x : (batch_size, longueur_sequence)
        x_emb = self.embedding(x)
        
        # out : tenseur contenant toutes les sorties temporelles
        # h_n : état caché final
        out, h_n = self.rnn(x_emb)
        
        # Pour une tâche de classification/prédiction, on prend souvent la sortie du dernier token
        out = self.fc(out[:, -1, :]) 
        return out


In [ ]:

# ==========================================
# 2. Implémentation d'un LSTM
# ==========================================
class ModeleLSTM(nn.Module):
    def __init__(self, taille_vocabulaire, dim_embedding, dim_cachee, dim_sortie):
        super(ModeleLSTM, self).__init__()
        self.dim_cachee = dim_cachee
        
        self.embedding = nn.Embedding(taille_vocabulaire, dim_embedding)
        
        # Couche LSTM au lieu de RNN
        self.lstm = nn.LSTM(dim_embedding, dim_cachee, batch_first=True)
        
        self.fc = nn.Linear(dim_cachee, dim_sortie)

    def forward(self, x):
        x_emb = self.embedding(x)
        
        # Le LSTM retourne la sortie complète, ainsi qu'un tuple (état caché final, état de la cellule final)
        out, (h_n, c_n) = self.lstm(x_emb)
        
        # Extraction de la sortie du dernier pas de temps
        out = self.fc(out[:, -1, :])
        return out


In [ ]:

# ==========================================
# 3. Implémentation d'un GRU
# ==========================================
class ModeleGRU(nn.Module):
    def __init__(self, taille_vocabulaire, dim_embedding, dim_cachee, dim_sortie):
        super(ModeleGRU, self).__init__()
        self.dim_cachee = dim_cachee
        
        self.embedding = nn.Embedding(taille_vocabulaire, dim_embedding)
        
        # Couche GRU
        self.gru = nn.GRU(dim_embedding, dim_cachee, batch_first=True)
        
        self.fc = nn.Linear(dim_cachee, dim_sortie)

    def forward(self, x):
        x_emb = self.embedding(x)
        
        # Le GRU simplifie le LSTM : il ne retourne que la sortie temporelle et l'état caché final
        out, h_n = self.gru(x_emb)
        
        out = self.fc(out[:, -1, :])
        return out


In [ ]:

# ==========================================
# Exemple d'instanciation et de test (Forward Pass)
# ==========================================
if __name__ == "__main__":
    # Hyperparamètres fictifs
    TAILLE_VOCAB = 10000
    DIM_EMBEDDING = 128
    DIM_CACHEE = 256
    DIM_SORTIE = 10  # par exemple, 10 classes ou tokens possibles
    
    # Création des modèles
    modele_rnn = ModeleRNNSimple(TAILLE_VOCAB, DIM_EMBEDDING, DIM_CACHEE, DIM_SORTIE).to(device)
    modele_lstm = ModeleLSTM(TAILLE_VOCAB, DIM_EMBEDDING, DIM_CACHEE, DIM_SORTIE).to(device)
    modele_gru = ModeleGRU(TAILLE_VOCAB, DIM_EMBEDDING, DIM_CACHEE, DIM_SORTIE).to(device)
    
    # Simulation d'un batch de données : 32 phrases de 20 tokens chacune
    batch_size = 32
    seq_length = 20
    entrees_simulees = torch.randint(0, TAILLE_VOCAB, (batch_size, seq_length)).to(device)
    
    # Test d'un passage en avant (Forward Pass) sur le GRU
    predictions = modele_gru(entrees_simulees)
    
    print(f"Forme des données en entrée : {entrees_simulees.shape}") # (32, 20)
    print(f"Forme des prédictions (GRU) : {predictions.shape}")      # (32, 10)